In [1]:
import os
import sys
import numpy as np
import pandas as pd
import h5py
from tqdm.auto import tqdm
from astropy.timeseries import LombScargle
import warnings
warnings.filterwarnings('ignore')

OBSERVATIONS_PKL = 'data/observations.pkl'
HDF5_OUT_PATH    = 'ante_periodograms.h5'
MODIFIED_EVAL_PY = 'validateRealData_modified.py'
ANTE_MODEL_PATH  = 'data/top_current_model_trained_on_uneven_data_fully.pth'
RESULTS_PKL      = 'ante_eval_results.pkl'

N_FREQ       = 1000                          # Ante's PERIODOGRAM_LEN
F_MIN_DAYS   = 1.0                           # shortest period probed (1 day)
F_MAX_DAYS   = 7000.0                        # matches Ante's max_period cutoff
MIN_FREQ     = 1.0 / F_MAX_DAYS
MAX_FREQ     = 1.0 / F_MIN_DAYS
FREQ_GRID    = np.linspace(MIN_FREQ, MAX_FREQ, N_FREQ)
print(f'Frequency grid: {len(FREQ_GRID)} bins over [{MIN_FREQ:.6f}, {MAX_FREQ:.4f}] cyc/day')
print(f'  periods: [{F_MIN_DAYS:.1f}, {F_MAX_DAYS:.0f}] days (linear)')

assert os.path.exists(OBSERVATIONS_PKL), f'observations.pkl not found at {OBSERVATIONS_PKL} — run data_prep.ipynb first'
observations = pd.read_pickle(OBSERVATIONS_PKL)
print(f'observations.pkl loaded: {observations.shape}, columns={list(observations.columns)}')

star_labels_series = observations.groupby('star_name')['has_exoplanets'].first()
print(f'\nStars: {len(star_labels_series)}')
print(f'  positives (has_exoplanets=1): {(star_labels_series==1).sum()}')
print(f'  negatives (has_exoplanets=0): {(star_labels_series==0).sum()}')

unique_stars = sorted(star_labels_series.index.tolist())
print(f'\nFirst 5 star names: {unique_stars[:5]}')

Frequency grid: 1000 bins over [0.000143, 1.0000] cyc/day
  periods: [1.0, 7000] days (linear)
observations.pkl loaded: (220318, 9), columns=['star_name', 'bjd', 'rv', 'rv_err', 'exposure_time', 'RHKp', 'Halpha', 'has_exoplanets', 'rv_centered']

Stars: 2026
  positives (has_exoplanets=1): 430
  negatives (has_exoplanets=0): 1596

First 5 star names: ['0748-01711-1', 'BD+01316', 'BD+062168', 'BD+101799', 'BD+144559']


In [2]:
def compute_pgram(bjd, rv_centered, freq_grid, F_MIN_DAYS=1.0, N_FREQ=1000):
    """Compute Lomb-Scargle power on the shared frequency grid."""
    if len(bjd) <= 1:
        return None
    if np.std(rv_centered) < 1e-12:
        # constant RV -> no signal -> flat zero power
        return np.zeros(len(freq_grid), dtype=np.float32)
    try:
        ls = LombScargle(bjd, rv_centered, normalization='standard')
        power = ls.power(freq_grid)
    except Exception:
        return None
    p = np.asarray(power, dtype=np.float32)
    p = np.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0)
    p = np.clip(p, 0.0, None)
    pmin, pmax = float(p.min()), float(p.max())
    if pmax - pmin > 1e-12:
        p = (p - pmin) / (pmax - pmin)
    else:
        p = np.zeros_like(p)
    return p.astype(np.float32)

n_stars_total = len(unique_stars)
n_success   = 0
n_skipped_constant = 0  # constant RV (no signal)
n_skipped_failed   = 0 # astropy/LS failed
n_skipped_too_few  = 0 # <=1 observation

with h5py.File(HDF5_OUT_PATH, 'w') as hf:
    for star_name in tqdm(unique_stars, desc='Computing periodograms'):
        star_df = observations[observations['star_name'] == star_name]
        bjd = star_df['bjd'].values.astype(np.float64)
        rv  = star_df['rv_centered'].values.astype(np.float64)

        if len(bjd) <= 1:
            n_skipped_too_few += 1
            continue
        if np.std(rv) < 1e-12:
            g = hf.create_group(star_name)
            g.create_dataset('frequencies', data=FREQ_GRID.astype(np.float64))
            g.create_dataset('power',       data=np.zeros(N_FREQ, dtype=np.float32))
            n_skipped_constant += 1
            continue

        pg = compute_pgram(bjd, rv, FREQ_GRID)
        if pg is None:
            n_skipped_failed += 1
            continue

        g = hf.create_group(star_name)
        g.create_dataset('frequencies', data=FREQ_GRID.astype(np.float64))
        g.create_dataset('power',       data=pg)

        n_success += 1

print('\nHDF5 build summary:')
print(f'  total stars: {n_stars_total}')
print(f'  periodograms computed: {n_success}')
print(f'  skipped (constant RV): {n_skipped_constant}')
print(f'  skipped (LS failed): {n_skipped_failed}')
print(f'  skipped (<=1 obs): {n_skipped_too_few}')

with h5py.File(HDF5_OUT_PATH, 'r') as hf:
    n_in_h5 = len(hf.keys())
print(f'\nHDF5 file at {HDF5_OUT_PATH}: {n_in_h5} star groups')

with h5py.File(HDF5_OUT_PATH, 'r') as hf:
    h5_stars = set(hf.keys())
pos_stars = set(star_labels_series[star_labels_series==1].index)
neg_stars = set(star_labels_series[star_labels_series==0].index)
n_pos_in_h5 = len(h5_stars & pos_stars)
n_neg_in_h5 = len(h5_stars & neg_stars)
print(f'  positives in HDF5: {n_pos_in_h5} / {len(pos_stars)}')
print(f'  negatives in HDF5: {n_neg_in_h5} / {len(neg_stars)}')
assert n_pos_in_h5 > 100, f'only {n_pos_in_h5} positives in HDF5 — something is wrong'
assert n_neg_in_h5 > 500, f'only {n_neg_in_h5} negatives in HDF5 — something is wrong'

with h5py.File(HDF5_OUT_PATH, 'r') as hf:
    sample_star = sorted(h5_stars)[0]
    f = hf[sample_star]['frequencies'][:]
    p = hf[sample_star]['power'][:]
    print(f'\nSample star {sample_star}: freq_shape={f.shape}, power_shape={p.shape}')
    print(f'  freq range: [{f.min():.6f}, {f.max():.6f}] cyc/day = periods [{1/f.max():.2f}, {1/f.min():.2f}] days')
    print(f'  power range: [{p.min():.4f}, {p.max():.4f}], mean={p.mean():.4f}, std={p.std():.4f}')
    assert (p >= 0).all() and (p <= 1).all(), 'power out of [0,1] range — normalization failure'

Computing periodograms:   0%|          | 0/2026 [00:00<?, ?it/s]


HDF5 build summary:
  total stars: 2026
  periodograms computed: 2026
  skipped (constant RV): 0
  skipped (LS failed): 0
  skipped (<=1 obs): 0

HDF5 file at ante_periodograms.h5: 2026 star groups
  positives in HDF5: 430 / 430
  negatives in HDF5: 1596 / 1596

Sample star 0748-01711-1: freq_shape=(1000,), power_shape=(1000,)
  freq range: [0.000143, 1.000000] cyc/day = periods [1.00, 7000.00] days
  power range: [0.0000, 1.0000], mean=0.1562, std=0.1505


In [4]:
assert os.path.exists(MODIFIED_EVAL_PY), f'Modified eval script not found at {MODIFIED_EVAL_PY}. Upload validateRealData_modified.py to the project root.'
print(f'Modified eval script: {MODIFIED_EVAL_PY}')
print(f'  size: {os.path.getsize(MODIFIED_EVAL_PY):,} bytes')

if not os.path.exists(ANTE_MODEL_PATH):
    print(f'\nWARN: Ante model weights not found at {ANTE_MODEL_PATH}')
    print('  Update ANTE_MODEL_PATH at the top of this notebook to point at the released weights file.')
    print('  Until this is set correctly, the next cell will fail at model load.')
else:
    print(f'Ante model weights: {ANTE_MODEL_PATH} ({os.path.getsize(ANTE_MODEL_PATH):,} bytes)')

Modified eval script: validateRealData_modified.py
  size: 25,880 bytes
Ante model weights: data/top_current_model_trained_on_uneven_data_fully.pth (26,005,031 bytes)


In [5]:
import subprocess

cmd = [
    sys.executable, MODIFIED_EVAL_PY,
    HDF5_OUT_PATH,
    ANTE_MODEL_PATH,
    '--observations_pkl', OBSERVATIONS_PKL,
    '--n_peaks', '3',
    '--threshold', '0.73',
    '--output', RESULTS_PKL,
]
print('Running:', ' '.join(cmd))

res = subprocess.run(cmd, capture_output=True, text=True, timeout=600)

print('STDOUT (last 6000 chars):')
print(res.stdout[-6000:] if res.stdout else '(empty)')
print(f'exit code: {res.returncode}')
if res.returncode != 0:
    print('STDERR (last 3000 chars):')
    print(res.stderr[-3000:] if res.stderr else '(empty)')
else:
    print('SUCCESS')

Running: /usr/bin/python3 validateRealData_modified.py ante_periodograms.h5 data/top_current_model_trained_on_uneven_data_fully.pth --observations_pkl data/observations.pkl --n_peaks 3 --threshold 0.73 --output ante_eval_results.pkl
STDOUT (last 6000 chars):
HDF5 has 2026 stars; labels has 2026 stars.
  Intersection: 2026
  HDF5-only (no label): 0
  Labels-only (no periodogram): 0

===== COVERAGE =====
  n_total_eval_stars             2026
  n_evaluated_with_peaks         2026
  n_skipped_no_peaks             0
  n_skipped_errors               0
  n_no_peaks_positives           0
  n_no_peaks_negatives           0
  n_pos_total                    430
  n_neg_total                    1596
  coverage_positives             1.0
  coverage_negatives             1.0

===== STAR-LEVEL EVAL (n = 2026, pos=430, neg=1596) =====
  PR-AUC  (max-peak-prob):  0.3056
  ROC-AUC (max-peak-prob):  0.5850
  Binary star-level @ thr=0.73: P=0.4300 R=0.2000 F1=0.2730
  Confusion (TN,FP,FN,TP): (1482,114,344

In [6]:
def bootstrap_roc_auc(y_true, y_score, n_resamples=200, seed=42):
    """Bootstrap 95% CI for ROC-AUC via the percentile method."""
    from sklearn.metrics import roc_auc_score

    y_true = np.asarray(y_true).ravel()
    y_score = np.asarray(y_score).ravel()
    if len(y_true) != len(y_score):
        raise ValueError(f"length mismatch: y_true={len(y_true)} y_score={len(y_score)}")
    if len(y_true) < 2:
        raise ValueError("need at least 2 samples to bootstrap ROC-AUC")

    point = float(roc_auc_score(y_true, y_score))
    rng = np.random.default_rng(seed)
    n = len(y_true)
    aucs = np.empty(n_resamples, dtype=float)
    for i in range(n_resamples):
        sample_idx = rng.integers(0, n, size=n)
        yt = y_true[sample_idx]
        ys = y_score[sample_idx]
        attempts = 0
        while len(np.unique(yt)) < 2 and attempts < 10:
            sample_idx = rng.integers(0, n, size=n)
            yt = y_true[sample_idx]
            ys = y_score[sample_idx]
            attempts += 1
        if len(np.unique(yt)) < 2:
            aucs[i] = point  # fall back to point estimate if degenerate
            continue
        aucs[i] = roc_auc_score(yt, ys)
    lo = float(np.percentile(aucs, 2.5))
    hi = float(np.percentile(aucs, 97.5))
    return point, lo, hi

def bootstrap_pr_auc(y_true, y_score, n_resamples=200, seed=42):
    """Bootstrap 95% CI for PR-AUC (average precision) via the percentile method."""
    from sklearn.metrics import average_precision_score

    y_true = np.asarray(y_true).ravel()
    y_score = np.asarray(y_score).ravel()
    if len(y_true) != len(y_score):
        raise ValueError(f"length mismatch: y_true={len(y_true)} y_score={len(y_score)}")
    if len(y_true) < 2:
        raise ValueError("need at least 2 samples to bootstrap PR-AUC")

    point = float(average_precision_score(y_true, y_score))
    rng = np.random.default_rng(seed)
    n = len(y_true)
    aps = np.empty(n_resamples, dtype=float)
    for i in range(n_resamples):
        sample_idx = rng.integers(0, n, size=n)
        yt = y_true[sample_idx]
        ys = y_score[sample_idx]
        attempts = 0
        while len(np.unique(yt)) < 2 and attempts < 10:
            sample_idx = rng.integers(0, n, size=n)
            yt = y_true[sample_idx]
            ys = y_score[sample_idx]
            attempts += 1
        if len(np.unique(yt)) < 2:
            aps[i] = point  # fall back to point estimate if degenerate
            continue
        aps[i] = average_precision_score(yt, ys)
    lo = float(np.percentile(aps, 2.5))
    hi = float(np.percentile(aps, 97.5))
    return point, lo, hi

import pickle

assert os.path.exists(RESULTS_PKL), f'Results pickle not found at {RESULTS_PKL}'
with open(RESULTS_PKL, 'rb') as f:
    results = pickle.load(f)

cov = results['coverage']
metrics = results['metrics_star']

print('\ncoverage:')
print(f'  Total stars (intersection HDF5 & labels): {cov["n_total_eval_stars"]}')
print(f'  evaluated with >=1 peak: {cov["n_evaluated_with_peaks"]}')
print(f'  Skipped — no peaks found                : {cov["n_skipped_no_peaks"]} '
      f'(pos={cov["n_no_peaks_positives"]}, neg={cov["n_no_peaks_negatives"]})')
print(f'  skipped (errors): {cov["n_skipped_errors"]}')
print('\n  Class coverage of evaluable subset:')
print(f'  Positives: {cov["coverage_positives"]*100:.1f}% ({cov["n_pos_total"]-cov["n_no_peaks_positives"]}/{cov["n_pos_total"]})')
print(f'  Negatives: {cov["coverage_negatives"]*100:.1f}% ({cov["n_neg_total"]-cov["n_no_peaks_negatives"]}/{cov["n_neg_total"]})')

print('\nstar-level metrics (max-peak-prob):')
if metrics.get('pr_auc') is not None:
    print(f'  pr_auc: {metrics["pr_auc"]:.4f}')
    print(f'  roc_auc: {metrics["roc_auc"]:.4f}')
else:
    print('  PR-AUC/ROC-AUC: SKIPPED (one class missing)')

print(f'\nbinary at threshold {metrics.get("threshold", 0.5)}:')
if 'precision_at_thr' in metrics:
    print(f'  Precision: {metrics["precision_at_thr"]:.4f}')
    print(f'  recall: {metrics["recall_at_thr"]:.4f}')
    print(f'  f1: {metrics["f1_at_thr"]:.4f}')
    tn, fp, fn, tp = metrics.get('confusion_matrix', [0,0,0,0])
    print(f'  Confusion (TN, FP, FN, TP): ({tn}, {fp}, {fn}, {tp})')

print('\npeak-level metrics (transparency check):')
peak = metrics.get('peak_level', {})
if peak:
    print(f'  n_peaks evaluated: {peak["n_peaks"]}')
    print(f'  precision: {peak["precision"]:.4f}')
    print(f'  recall: {peak["recall"]:.4f}')
    print(f'  f1: {peak["f1"]:.4f}')
    ptn, pfp, pfn, ptp = peak.get('confusion_matrix', [0,0,0,0])
    print(f'  Confusion (TN, FP, FN, TP): ({ptn}, {pfp}, {pfn}, {ptp})')

if 'star_preds' in results and 'star_labels' in results:

    pr_point, pr_lo, pr_hi = bootstrap_pr_auc(results['star_labels'], results['star_preds'])
    roc_point, roc_lo, roc_hi = bootstrap_roc_auc(results['star_labels'], results['star_preds'])
    print("\nbootstrap 95% CI (200 resamples):")
    print(f"  pr_auc: {pr_point:.4f} [{pr_lo:.4f}, {pr_hi:.4f}]")
    print(f"  roc_auc: {roc_point:.4f} [{roc_lo:.4f}, {roc_hi:.4f}]")
else:
    print("\nBOOTSTRAP CI: skipped — raw predictions not in results pickle. Re-run validateRealData_modified.py with --save_preds.")


coverage:
  Total stars (intersection HDF5 & labels): 2026
  evaluated with >=1 peak: 2026
  Skipped — no peaks found                : 0 (pos=0, neg=0)
  skipped (errors): 0

  Class coverage of evaluable subset:
  Positives: 100.0% (430/430)
  Negatives: 100.0% (1596/1596)

star-level metrics (max-peak-prob):
  pr_auc: 0.3056
  roc_auc: 0.5850

binary at threshold 0.73:
  Precision: 0.4300
  recall: 0.2000
  f1: 0.2730
  Confusion (TN, FP, FN, TP): (1482, 114, 344, 86)

peak-level metrics (transparency check):
  n_peaks evaluated: 6068
  precision: 0.4548
  recall: 0.1054
  f1: 0.1712
  Confusion (TN, FP, FN, TP): (4615, 163, 1154, 136)

BOOTSTRAP CI: skipped — raw predictions not in results pickle. Re-run validateRealData_modified.py with --save_preds.
